In [ ]:
import pandas as pd
import numpy as np
import re
import spacy
import nltk
from nltk.corpus import stopwords

# 1. Setup NLTK & spaCy
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

# 2. Load Kaggle Dataset (UPDATED PATH)
df = pd.read_csv('experiments/data/train.csv')
print(f"Original Dataset Shape: {df.shape}")

# 3. Create Binary Target ('is_toxic')
categories = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
df['is_toxic'] = df[categories].max(axis=1)

# 4. Define Cleaning Pipeline
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    
    doc = nlp(text)
    tokens = [
        token.lemma_ for token in doc 
        if token.text not in stop_words and not token.is_space
    ]
    return " ".join(tokens)

print("Preprocessing dataset (lemmatization & cleaning)...")

# 5. Sample the data for faster processing on your machine
toxic_df = df[df['is_toxic'] == 1]
clean_df = df[df['is_toxic'] == 0].sample(n=len(toxic_df) * 2, random_state=42)
sampled_df = pd.concat([toxic_df, clean_df]).sample(frac=1, random_state=42).reset_index(drop=True)

# 6. Apply cleaning (This will take a minute or two)
sampled_df['cleaned_text'] = sampled_df['comment_text'].apply(clean_text)

# Drop any blank rows resulting from cleaning
sampled_df = sampled_df[sampled_df['cleaned_text'].str.strip() != '']

# 7. Save preprocessed dataset (UPDATED PATH)
sampled_df.to_csv('experiments/data/cleaned_train.csv', index=False)
print(f"Success! Preprocessed {len(sampled_df)} records saved to 'experiments/data/cleaned_train.csv'")  

[nltk_data] Downloading package stopwords to C:\Users\Aayush
[nltk_data]     Chauhan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Original Dataset Shape: (159571, 8)
Preprocessing dataset (lemmatization & cleaning)...
Success! Preprocessed 48666 records saved to 'experiments/data/cleaned_train.csv'
